<div style="background:linear-gradient(135deg,#0d1117,#13243b,#0f3a5f);padding:44px 38px;border-radius:16px;color:#f0f6fc;font-family:'Segoe UI',sans-serif;border:1px solid #30363d;">
  <div style="font-size:.8em;letter-spacing:3px;opacity:.65;text-transform:uppercase;margin-bottom:10px;">Seed Variance · Noise Floor · Reproducibility</div>
  <h1 style="font-size:2.0em;margin:0 0 12px 0;font-weight:700;line-height:1.25;color:#f0f6fc !important;">Çoklu Tohumla Gürültü Tabanının Ölçülmesi</h1>
  <h2 style="font-size:1.05em;font-weight:300;opacity:.82;margin:0 0 22px 0;line-height:1.5;color:#f0f6fc !important;">Kollar arası farkın, aynı kolun kendi koşumları arasındaki değişkenlikten büyük olup olmadığı</h2>
  <hr style="border:0;border-top:1px solid #30363d;margin:0 0 18px 0;">
  <div style="display:grid;grid-template-columns:1fr 1fr;gap:12px;font-size:.88em;opacity:.88;">
    <div><b>Sabit:</b> veri bölmesi (tohum 42) ve dış örneklem</div>
    <div><b>Değişken:</b> yalnızca eğitim tohumu</div>
    <div><b>Süre:</b> ~7 saat (2 tohum × 3 kol)</div>
    <div><b>Ek analiz:</b> tohum topluluğu (ensemble)</div>
  </div>
  <div style="margin-top:20px;padding:13px 16px;background:rgba(56,139,253,.10);border-left:4px solid #388bfd;border-radius:4px;font-size:.86em;line-height:1.55;">
    <b>Gerekçe:</b> Ablasyonda C − A = +0,010 bulundu. Aynı yapılandırmanın iki bağımsız koşumu arasındaki fark ise 0,0036 ölçüldü — yani gürültü, aranan etkinin ~%35'i. Tek tohumla bu soru kapanmaz. Bu notebook her kolu birden çok tohumla eğitip <b>kol-içi değişkenliği (gürültü tabanı)</b> ile <b>kollar arası farkı</b> aynı ölçekte karşılaştırır.
  </div>
</div>

## Tasarım: neyin değiştiği, neyin değişmediği

Bu ayrım kritiktir ve yanlış kurulursa tüm karşılaştırma çöker.

| | Tohum | Neden |
|---|---|---|
| **Veri bölmesi** | **42, sabit** | Tohum değişirse her koşum farklı train/val/test görür; kollar ve tohumlar karşılaştırılamaz hale gelir |
| **Dış örneklem seçimi** | **42, sabit** | Aynı RSNA/NIH görüntüleri değerlendirilmeli |
| **Eğitim** (ağırlık başlangıcı, yığın sırası, artırma) | **değişken** | Ölçmek istediğimiz belirsizlik kaynağı budur |

Bölmenin doğruluğu, önceki ablasyon koşumunun yazdırdığı **MD5 imzalarıyla** otomatik
denetlenir. İmza tutmuyorsa notebook durur.

### Ne raporlanacak

1. **Gürültü tabanı:** her kolun kendi tohumları arasındaki AUC standart sapması.
2. **Kollar arası fark:** tohum-eşleştirilmiş ΔAUC (her tohumda C − A, sonra ortalama ± ss).
3. **Karar:** kollar arası fark, gürültü tabanının kaç katı?
4. **Topluluk:** tohumların olasılık ortalaması. Topluluk, eğitim gürültüsünü büyük ölçüde
   giderir; topluluklar arası DeLong testi ön-işleme etkisini en temiz haliyle verir.

### Çalıştırma süresi ve strateji

Kol başına eğitim ~67 dk. Varsayılan `TRAIN_SEEDS = [1337, 2024]` (2 yeni tohum × 3 kol
= 6 eğitim ≈ **6,7 saat**) artı ön-işleme ~15 dk ve dış çıkarım ~10 dk → **≈ 7,2 saat**.
Kaggle'ın 12 saatlik sınırına sığar.

Tohum 42 sonuçları önceki ablasyon koşumundan **birleştirilir**: o koşumun
`ablation_predictions.npz` dosyasını girdi olarak eklerseniz analiz 3 tohumla yapılır.
Eklemezseniz 2 tohumla devam eder (gürültü tahmini zayıflar).

> Süre riskliyse `ARMS = ["raw", "lung"]` yapın: 4 eğitim ≈ 4,5 saat. B kolu düşer,
> ancak asıl karşılaştırma (C − A) korunur.

In [ ]:
import os, sys, gc, io, json, glob, time, math, random, zipfile, hashlib, warnings, types
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image
from collections import OrderedDict, Counter, defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, models

from sklearn.metrics import (roc_curve, auc, average_precision_score, f1_score,
                             confusion_matrix, brier_score_loss)
from scipy import stats as sp_stats

ARM_LABEL = OrderedDict([
    ("raw",  "A · Segmentasyonsuz"),
    ("roi",  "B · Yalnizca RoI kirpma"),
    ("lung", "C · Maske + RoI (onerilen)"),
])
ARM_COLOR = {"raw": "#B04A1E", "roi": "#C2900A", "lung": "#0D8FA2"}

plt.rcParams.update({
    "figure.dpi": 130, "figure.facecolor": "white", "savefig.facecolor": "white",
    "font.size": 10, "axes.titlesize": 11, "axes.labelsize": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#8A9499", "axes.labelcolor": "#1B2327", "text.color": "#1B2327",
    "xtick.color": "#5A686F", "ytick.color": "#5A686F",
    "grid.color": "#D8DFE1", "grid.linewidth": 0.7, "legend.frameon": False,
})
WORK = "/kaggle/working"


import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets
from sklearn.metrics import classification_report

# ----------------------------- AYARLAR -----------------------------
SPLIT_SEED   = 42                 # SABIT - bolme ve dis orneklem icin
TRAIN_SEEDS  = [1337, 2024]       # YENI egitim tohumlari (42 onceki kosumdan gelir)
ARMS         = ["raw", "roi", "lung"]
EPOCHS       = 15
BATCH_SIZE   = 16
K_EXT        = 400                # dis setlerden sinif basina - ablasyonla AYNI
N_BOOT       = 2000
QUICK_TEST   = False
SAVE_CHECKPOINTS = False          # 6 x 343 MB - varsayilan kapali
# --------------------------------------------------------------------

# Onceki ablasyon kosumunun bolme imzalari - otomatik denetim
EXPECTED_FP = {"train": "ccf23597ec992137",
               "val":   "779ac7f6455a7ac8",
               "test":  "3a25cbb40ab471d7"}

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

def seed_worker(worker_id):
    ws = torch.initial_seed() % 2**32 + worker_id
    np.random.seed(ws % 2**32); random.seed(ws % 2**32)

torch.backends.cudnn.benchmark = False
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CONFIG = {
    "img_size": 224, "batch_size": BATCH_SIZE, "epochs": EPOCHS,
    "lr": 1e-4, "weight_decay": 1e-2, "dropout": 0.1, "num_classes": 2,
    "mean": [0.4769, 0.4769, 0.4769], "std": [0.2414, 0.2414, 0.2414],
    "orig_short_max": 512, "mask_dilate_frac": 0.025, "roi_pad_frac": 0.05,
    "mask_feather": 9, "fill_mode": "mean", "raw_mode": "resize",
}

_cache_base = "/kaggle/temp/seeds" if os.path.isdir("/kaggle/temp") else "/kaggle/working/_seed_cache"
CACHE_ROOT = _cache_base + ("_quick" if QUICK_TEST else "")
os.makedirs(CACHE_ROOT, exist_ok=True)

print("=" * 66)
print(f"  Cihaz        : {device} | Torch {torch.__version__}")
print(f"  Bolme tohumu : {SPLIT_SEED} (SABIT)")
print(f"  Egitim tohum : {TRAIN_SEEDS}")
print(f"  Kollar       : {ARMS}")
print(f"  Tahmini sure : {len(TRAIN_SEEDS)*len(ARMS)*67/60:.1f} sa (egitim)")
print("=" * 66)

In [ ]:
# ── Girdiler ─────────────────────────────────────────────────────────────
INPUT = "/kaggle/input"

def find_classification_root(root=INPUT):
    hits = []
    for r, dirs, _ in os.walk(root):
        if os.path.basename(r) == "train" and {"NORMAL", "PNEUMONIA"} <= set(dirs):
            hits.append(os.path.dirname(r))
    if not hits:
        return None
    hits.sort(key=lambda p: (0 if "balanced" in p.lower() else 1, len(p)))
    return hits[0]

def find_dir_with(fname, root=INPUT):
    for r, _, files in os.walk(root):
        if fname in files:
            return r
    return None

BASE_PATH = find_classification_root()
assert BASE_PATH is not None, "Siniflandirma veri kumesi bulunamadi."
TRAIN_PATH = os.path.join(BASE_PATH, "train")
VAL_PATH   = os.path.join(BASE_PATH, "val")
TEST_PATH  = os.path.join(BASE_PATH, "test")
RSNA_BASE  = find_dir_with("stage_2_detailed_class_info.csv")
NIH_BASE   = find_dir_with("Data_Entry_2017.csv")

# Onceki ablasyon tahminleri (tohum 42) - opsiyonel
PREV = None
for pat in ("ablation_predictions.npz", "ablation_predictions.zip"):
    hits = glob.glob(os.path.join(INPUT, "**", pat), recursive=True)
    if hits:
        try:
            z = zipfile.ZipFile(hits[0])
            PREV = {n[:-4]: np.load(io.BytesIO(z.read(n))) for n in z.namelist() if n.endswith(".npy")}
            print(f"Onceki kosum bulundu ({os.path.basename(hits[0])}): "
                  f"{len(PREV)} dizi -> tohum 42 birlestirilecek")
        except Exception as e:
            print("Onceki kosum okunamadi:", e)
        break
if PREV is None:
    # Kaggle yuklenen zip'i acmis olabilir: gevsek .npy dosyalarini ara
    loose = glob.glob(os.path.join(INPUT, "**", "*__y_prob.npy"), recursive=True)
    if loose:
        d = os.path.dirname(loose[0])
        PREV = {os.path.basename(f)[:-4]: np.load(f)
                for f in glob.glob(os.path.join(d, "*.npy"))}
        print(f"Onceki kosum (acilmis .npy) bulundu: {len(PREV)} dizi -> tohum 42 birlestirilecek")
if PREV is None:
    print("Onceki kosum bulunamadi -> yalnizca yeni tohumlarla devam edilecek.")

print(f"\n  Siniflandirma : {BASE_PATH}")
print(f"  RSNA          : {RSNA_BASE}")
print(f"  NIH           : {NIH_BASE}")

In [ ]:
# ── Bolme: SPLIT_SEED ile birebir yeniden uretim + imza denetimi ─────────
def file_sig(p):
    return f"{os.path.basename(p)}_{os.path.getsize(p) if os.path.exists(p) else 0}"

def split_fingerprint(samples):
    key = "|".join(sorted(os.path.basename(p) for p, _ in samples))
    return hashlib.md5(key.encode("utf-8")).hexdigest()[:16]

set_seed(SPLIT_SEED)

train_if = datasets.ImageFolder(TRAIN_PATH)
val_if   = datasets.ImageFolder(VAL_PATH)
test_if  = datasets.ImageFolder(TEST_PATH)

CLASS_NAMES  = train_if.classes
CLASS_TO_IDX = train_if.class_to_idx
IDX_TO_CLASS = {v: k for k, v in CLASS_TO_IDX.items()}
PNEU_IDX     = CLASS_TO_IDX["PNEUMONIA"]

train_samples = list(train_if.samples)
pool = list(val_if.samples) + list(test_if.samples)
train_sigs = set(file_sig(p) for p, _ in train_samples)
pool = [(p, l) for p, l in pool if file_sig(p) not in train_sigs]

idx_n, idx_p = CLASS_TO_IDX["NORMAL"], CLASS_TO_IDX["PNEUMONIA"]
normal_pool = [(p, l) for p, l in pool if l == idx_n]
pneum_pool  = [(p, l) for p, l in pool if l == idx_p]
random.shuffle(normal_pool); random.shuffle(pneum_pool)
per = min(len(normal_pool), len(pneum_pool)) // 2
val_samples  = normal_pool[:per]      + pneum_pool[:per]
test_samples = normal_pool[per:2*per] + pneum_pool[per:2*per]
random.shuffle(val_samples); random.shuffle(test_samples)

splits_raw = {"train": train_samples, "val": val_samples, "test": test_samples}
if QUICK_TEST:
    rq = random.Random(SPLIT_SEED)
    splits_raw = {k: rq.sample(v, min(len(v), 240)) for k, v in splits_raw.items()}

print("=" * 72)
print("  BOLME DENETIMI")
print("=" * 72)
allok = True
for name in ["train", "val", "test"]:
    fp = split_fingerprint(splits_raw[name])
    exp = EXPECTED_FP[name]
    ok = (fp == exp) or QUICK_TEST
    allok &= ok
    print(f"  {name.upper():<6}: {len(splits_raw[name]):>5} kayit | imza {fp} "
          f"| beklenen {exp} | {'ESLESTI' if fp == exp else 'FARKLI'}")
print("=" * 72)
if QUICK_TEST:
    print("  QUICK_TEST etkin - imza denetimi atlandi.")
else:
    assert allok, ("Bolme imzalari onceki kosumla eslesmiyor! Ayni veri kumesi "
                   "surumunun ekli oldugundan emin olun; aksi halde tohumlar arasi "
                   "karsilastirma gecersizdir.")
    print("  Bolme onceki ablasyon kosumuyla BIREBIR ayni.")

In [ ]:
import transformers
from transformers import AutoModel

print("ianpan/chest-x-ray-basic yukleniyor... (transformers", transformers.__version__, ")")

def _load_seg():
    return AutoModel.from_pretrained("ianpan/chest-x-ray-basic",
                                     trust_remote_code=True).to(device).eval()

_orig_finalize = getattr(transformers.modeling_utils.PreTrainedModel,
                         "_finalize_model_loading", None)
try:
    if _orig_finalize is not None:
        def _safe_finalize(model, *a, **k):
            if not hasattr(model, "all_tied_weights_keys"):
                model.all_tied_weights_keys = {}
            return _orig_finalize(model, *a, **k)
        transformers.modeling_utils.PreTrainedModel._finalize_model_loading = _safe_finalize
    seg_model = _load_seg()
except Exception as e:
    if "all_tied_weights_keys" in str(e):
        transformers.modeling_utils.PreTrainedModel.all_tied_weights_keys = {}
        seg_model = _load_seg()
    else:
        raise
finally:
    if _orig_finalize is not None:
        transformers.modeling_utils.PreTrainedModel._finalize_model_loading = _orig_finalize

print("Segmentasyon modeli hazir.")

In [ ]:
def load_image_any(path, short_max):
    '''PNG/JPG/DICOM -> (rgb_u8, gray_u8); kisa kenari short_max'a indirir.'''
    ext = os.path.splitext(path)[1].lower()
    if ext == ".dcm":
        import pydicom
        dcm = pydicom.dcmread(path)
        arr = dcm.pixel_array.astype(np.float32)
        arr -= arr.min()
        if arr.max() > 0:
            arr /= arr.max()
        if getattr(dcm, "PhotometricInterpretation", "MONOCHROME2") == "MONOCHROME1":
            arr = 1.0 - arr
        pil = Image.fromarray((arr * 255).astype(np.uint8)).convert("RGB")
    else:
        pil = Image.open(path).convert("RGB")
    W0, H0 = pil.size
    short = min(W0, H0)
    if short > short_max:
        s = short_max / short
        pil = pil.resize((int(round(W0 * s)), int(round(H0 * s))), Image.BILINEAR)
    return np.asarray(pil).astype(np.uint8), np.asarray(pil.convert("L"))


@torch.inference_mode()
def lung_mask_ianpan(gray_u8, out_hw):
    '''Sag + sol akciger maskesi; kalp (sinif 3) dislanir.'''
    x = seg_model.preprocess(gray_u8)
    x = torch.from_numpy(x).unsqueeze(0).unsqueeze(0).float().to(device)
    logits = seg_model(x)["mask"]
    logits = F.interpolate(logits, size=out_hw, mode="bilinear", align_corners=False)
    pred = logits.argmax(dim=1)[0].cpu().numpy()
    return ((pred == 1) | (pred == 2)).astype(np.uint8)


def _center_square(img, S, interp):
    H, W = img.shape[:2]
    s = 256.0 / min(H, W)
    rz = cv2.resize(img, (max(S, int(round(W * s))), max(S, int(round(H * s)))), interpolation=interp)
    h2, w2 = rz.shape[:2]
    y0, x0 = (h2 - S) // 2, (w2 - S) // 2
    return rz[y0:y0 + S, x0:x0 + S]


def prep_arms(rgb_u8, lung_u8, cfg):
    '''Tek segmentasyon cikisindan uc kolun girdisi. Ablasyon notebook'u ile birebir ayni.'''
    H, W = lung_u8.shape
    short = min(H, W)
    S = cfg["img_size"]
    out = {}

    if cfg.get("raw_mode", "resize") == "centercrop":
        a_img = _center_square(rgb_u8, S, cv2.INTER_AREA)
        a_msk = _center_square(lung_u8, S, cv2.INTER_NEAREST)
    else:
        a_img = cv2.resize(rgb_u8, (S, S), interpolation=cv2.INTER_AREA)
        a_msk = cv2.resize(lung_u8, (S, S), interpolation=cv2.INTER_NEAREST)
    out["raw"] = (a_img, a_msk.astype(np.uint8), True)

    if lung_u8.sum() < 1:
        fb = cv2.resize(rgb_u8, (S, S), interpolation=cv2.INTER_AREA)
        ones = np.ones((S, S), np.uint8)
        out["roi"] = (fb, ones, False)
        out["lung"] = (fb, ones, False)
        return out

    dil = max(1, int(round(short * cfg["mask_dilate_frac"])))
    kern = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * dil + 1, 2 * dil + 1))
    mask_d = cv2.dilate(lung_u8, kern, iterations=1)

    soft = mask_d.astype(np.float32)
    f = cfg["mask_feather"]
    if f and f >= 3:
        if f % 2 == 0:
            f += 1
        soft = cv2.GaussianBlur(soft, (f, f), 0)
    soft = np.clip(soft, 0.0, 1.0)[..., None]

    fill = (np.array([m * 255.0 for m in cfg["mean"]], dtype=np.float32)
            if cfg["fill_mode"] == "mean" else np.zeros(3, dtype=np.float32))
    masked = (rgb_u8.astype(np.float32) * soft +
              fill[None, None, :] * (1.0 - soft)).clip(0, 255).astype(np.uint8)

    ys, xs = np.where(mask_d > 0)
    y0, y1 = int(ys.min()), int(ys.max())
    x0, x1 = int(xs.min()), int(xs.max())
    pad = int(round(short * cfg["roi_pad_frac"]))
    y0 = max(0, y0 - pad); x0 = max(0, x0 - pad)
    y1 = min(H - 1, y1 + pad); x1 = min(W - 1, x1 + pad)

    bh, bw = (y1 - y0 + 1), (x1 - x0 + 1)
    side = min(max(bh, bw), min(H, W))
    cy, cx = (y0 + y1) // 2, (x0 + x1) // 2
    ty0 = max(0, cy - side // 2); tx0 = max(0, cx - side // 2)
    ty1 = min(H, ty0 + side);     tx1 = min(W, tx0 + side)
    ty0 = max(0, ty1 - side);     tx0 = max(0, tx1 - side)

    m224 = cv2.resize(mask_d[ty0:ty1, tx0:tx1], (S, S),
                      interpolation=cv2.INTER_NEAREST).astype(np.uint8)
    out["roi"]  = (cv2.resize(rgb_u8[ty0:ty1, tx0:tx1], (S, S),
                              interpolation=cv2.INTER_AREA), m224, True)
    out["lung"] = (cv2.resize(masked[ty0:ty1, tx0:tx1], (S, S),
                              interpolation=cv2.INTER_AREA), m224, True)
    return out

print("On-isleme hatti hazir (ablasyon notebook'u ile birebir ayni).")

In [ ]:
# ── Onbellek: tohumdan bagimsiz, bir kez kurulur ─────────────────────────
def cache_paths(arm, split, cls, stem):
    return (os.path.join(CACHE_ROOT, arm, split, cls, stem),
            os.path.join(CACHE_ROOT, arm, split + "_mask", cls, stem))

def build_cache(split, samples, cfg, log_every=1000):
    for arm in ARMS:
        for cls in CLASS_NAMES:
            os.makedirs(os.path.join(CACHE_ROOT, arm, split, cls), exist_ok=True)
            os.makedirs(os.path.join(CACHE_ROOT, arm, split + "_mask", cls), exist_ok=True)
    n_ok = n_fb = n_err = 0
    t0 = time.time()
    for i, (path, label) in enumerate(samples):
        cls = IDX_TO_CLASS[label]
        stem = f"{i:06d}_{os.path.splitext(os.path.basename(path))[0]}.png"
        if all(os.path.exists(cache_paths(a, split, cls, stem)[0]) for a in ARMS):
            n_ok += 1; continue
        try:
            rgb, gray = load_image_any(path, cfg["orig_short_max"])
            lung = lung_mask_ianpan(gray, gray.shape)
            arms = prep_arms(rgb, lung, cfg)
            for arm in ARMS:
                img, msk, used = arms[arm]
                p_img, p_msk = cache_paths(arm, split, cls, stem)
                Image.fromarray(img).save(p_img)
                Image.fromarray((msk * 255).astype(np.uint8)).save(p_msk)
            n_ok += 1; n_fb += (0 if arms["lung"][2] else 1)
        except Exception as e:
            n_err += 1
            if n_err <= 3:
                print(f"    hata ({os.path.basename(path)}): {e}")
        if (i + 1) % log_every == 0:
            print(f"    [{split}] {i+1}/{len(samples)} ({time.time()-t0:.0f} sn)")
    print(f"  [{split}] tamam: {n_ok} | geri cekilme={n_fb} | hata={n_err} | {time.time()-t0:.0f} sn")

print("Onbellek kuruluyor (tum tohumlar bunu paylasir)...\n")
for name in ["train", "val", "test"]:
    build_cache(name, splits_raw[name], CONFIG)
print(f"\nOnbellek hazir: {CACHE_ROOT}")

In [ ]:
# ── Egitim yapi taslari ──────────────────────────────────────────────────
data_transforms = {
    "train": transforms.Compose([
        transforms.Resize((CONFIG["img_size"], CONFIG["img_size"])),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=8),
        transforms.RandomAffine(degrees=0, translate=(0.04, 0.04), scale=(0.95, 1.05)),
        transforms.ColorJitter(brightness=0.15, contrast=0.15),
        transforms.ToTensor(),
        transforms.Normalize(CONFIG["mean"], CONFIG["std"]),
    ]),
    "eval": transforms.Compose([
        transforms.Resize((CONFIG["img_size"], CONFIG["img_size"])),
        transforms.ToTensor(),
        transforms.Normalize(CONFIG["mean"], CONFIG["std"]),
    ]),
}
eval_tf = data_transforms["eval"]


def make_loaders(arm, seed):
    ds, dl = {}, {}
    for split in ["train", "val", "test"]:
        tf = data_transforms["train" if split == "train" else "eval"]
        ds[split] = datasets.ImageFolder(os.path.join(CACHE_ROOT, arm, split), tf)
        assert ds[split].classes == CLASS_NAMES
        g = torch.Generator(); g.manual_seed(seed)
        dl[split] = DataLoader(ds[split], batch_size=CONFIG["batch_size"],
                               shuffle=(split == "train"), num_workers=2, pin_memory=True,
                               generator=g if split == "train" else None,
                               worker_init_fn=seed_worker)
    return ds, dl


def build_vit(fine_tune_last_n=2):
    m = models.vit_b_16(weights=models.ViT_B_16_Weights.IMAGENET1K_V1)
    for p in m.parameters():
        p.requires_grad = False
    for p in m.encoder.layers[-fine_tune_last_n:].parameters():
        p.requires_grad = True
    for p in m.encoder.ln.parameters():
        p.requires_grad = True
    in_f = m.heads.head.in_features
    m.heads.head = nn.Sequential(nn.Dropout(CONFIG["dropout"]),
                                 nn.Linear(in_f, CONFIG["num_classes"]))
    return m.to(device)


def run_epoch(model, loader, criterion, optimizer=None):
    train = optimizer is not None
    model.train() if train else model.eval()
    loss_sum, preds, labels, probs = 0.0, [], [], []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            if train:
                optimizer.zero_grad()
            out = model(imgs)
            loss = criterion(out, lbls)
            if train:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
            loss_sum += loss.item() * imgs.size(0)
            probs.extend(torch.softmax(out, 1)[:, 1].detach().cpu().numpy())
            preds.extend(out.argmax(1).cpu().numpy())
            labels.extend(lbls.cpu().numpy())
    N = len(loader.dataset)
    return (loss_sum / N, f1_score(labels, preds, zero_division=0),
            float((np.array(preds) == np.array(labels)).mean()),
            np.array(labels), np.array(preds), np.array(probs))


def train_one(arm, seed, epochs):
    set_seed(seed)                       # <-- yalnizca EGITIM rastgeleligi degisir
    ds, dl = make_loaders(arm, seed)
    model = build_vit(2)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                            lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
    sched = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    best_f1, best_wt, t0 = 0.0, None, time.time()
    print(f"\n{'-'*66}\n  kol={arm}  tohum={seed}  ({epochs} epoch)\n{'-'*66}")
    for ep in range(epochs):
        trl, trf, _, _, _, _ = run_epoch(model, dl["train"], criterion, optimizer)
        vll, vlf, vla, _, _, _ = run_epoch(model, dl["val"], criterion, None)
        sched.step()
        if vlf > best_f1:
            best_f1 = vlf
            best_wt = {k: v.detach().clone() for k, v in model.state_dict().items()}
        if (ep + 1) % 5 == 0 or ep == epochs - 1:
            print(f"    Ep{ep+1:3d} | TR F1 {trf:.4f} | VAL F1 {vlf:.4f} acc {vla:.4f}")
    if best_wt:
        model.load_state_dict(best_wt)
    print(f"    -> en iyi Val F1 = {best_f1:.4f}  ({time.time()-t0:.0f} sn)")
    return model, best_f1, ds, dl

In [ ]:
# ── Tum (tohum, kol) kombinasyonlarinin egitimi ──────────────────────────
EPOCHS_RUN = 2 if QUICK_TEST else EPOCHS
RUNS = {}          # RUNS[(seed, arm)] = model
BESTF1, DSETS, DLOAD = {}, {}, {}

t_all = time.time()
for seed in TRAIN_SEEDS:
    for arm in ARMS:
        m, b, ds, dl = train_one(arm, seed, EPOCHS_RUN)
        RUNS[(seed, arm)] = m
        BESTF1[(seed, arm)] = b
        DSETS[(seed, arm)] = ds
        DLOAD[(seed, arm)] = dl
        gc.collect(); torch.cuda.empty_cache()

print(f"\n{'='*66}")
print(f"  {len(RUNS)} egitim tamamlandi ({(time.time()-t_all)/60:.1f} dk)")
for (s, a), v in BESTF1.items():
    print(f"    tohum {s} · kol {a:<5} en iyi Val F1 = {v:.4f}")
print("=" * 66)

In [ ]:
# ── Ic test + dis dogrulama, tum kosumlar ────────────────────────────────
criterion = nn.CrossEntropyLoss()
PRED = defaultdict(dict)      # PRED[dataset][(seed, arm)] = {y_true, y_prob}

for (seed, arm), model in RUNS.items():
    _, _, _, yt, _, pr = run_epoch(model, DLOAD[(seed, arm)]["test"], criterion, None)
    PRED["internal"][(seed, arm)] = {"y_true": yt, "y_prob": pr}
print("Ic test cikarimi tamam.")


def build_rsna_items(base, k, seed=SPLIT_SEED):
    info = pd.read_csv(os.path.join(base, "stage_2_detailed_class_info.csv")).drop_duplicates("patientId")
    img_dir = os.path.join(base, "stage_2_train_images")
    pos = info[info["class"] == "Lung Opacity"]["patientId"].tolist()
    neg = info[info["class"] == "Normal"]["patientId"].tolist()
    rng = random.Random(seed); rng.shuffle(pos); rng.shuffle(neg)
    items  = [(os.path.join(img_dir, p + ".dcm"), PNEU_IDX)     for p in pos[:k]]
    items += [(os.path.join(img_dir, p + ".dcm"), 1 - PNEU_IDX) for p in neg[:k]]
    return items

def build_nih_items(base, k, seed=SPLIT_SEED):
    df = pd.read_csv(os.path.join(base, "Data_Entry_2017.csv"))
    labels = df["Finding Labels"].astype(str)
    is_pneu   = labels.apply(lambda s: "Pneumonia" in s.split("|"))
    is_normal = labels.apply(lambda s: s.strip() == "No Finding")
    index = {}
    for r, _, files in os.walk(base):
        for f in files:
            if f.lower().endswith(".png"):
                index[f] = os.path.join(r, f)
    def paths_for(mask_):
        return [index[n] for n in df[mask_]["Image Index"].tolist() if n in index]
    pos, neg = paths_for(is_pneu), paths_for(is_normal)
    rng = random.Random(seed); rng.shuffle(pos); rng.shuffle(neg)
    items  = [(p, PNEU_IDX)     for p in pos[:k]]
    items += [(p, 1 - PNEU_IDX) for p in neg[:k]]
    return items


KE = 60 if QUICK_TEST else K_EXT
EXTERNAL = []
if RSNA_BASE: EXTERNAL.append(("RSNA", build_rsna_items(RSNA_BASE, KE)))
if NIH_BASE:  EXTERNAL.append(("NIH",  build_nih_items(NIH_BASE, KE)))


@torch.inference_mode()
def external_all_runs(name, items):
    '''Goruntu basina TEK segmentasyon; tum (tohum, kol) modelleri ayni girdiden gecer.'''
    for m in RUNS.values():
        m.eval()
    y_true, probs = [], {k: [] for k in RUNS}
    t0 = time.time()
    for i, (path, label) in enumerate(items):
        try:
            rgb, gray = load_image_any(path, CONFIG["orig_short_max"])
            lung = lung_mask_ianpan(gray, gray.shape)
            arms = prep_arms(rgb, lung, CONFIG)
        except Exception:
            continue
        y_true.append(label)
        xs = {a: eval_tf(Image.fromarray(arms[a][0])).unsqueeze(0).to(device) for a in ARMS}
        for (seed, arm), m in RUNS.items():
            probs[(seed, arm)].append(torch.softmax(m(xs[arm]), 1)[0, PNEU_IDX].item())
        if (i + 1) % 200 == 0:
            print(f"    [{name}] {i+1}/{len(items)} ({time.time()-t0:.0f} sn)")
    print(f"  [{name}] bitti: {len(y_true)} goruntu | {time.time()-t0:.0f} sn")
    yt = np.array(y_true)
    return {k: {"y_true": yt, "y_prob": np.array(probs[k])} for k in RUNS}

for name, items in EXTERNAL:
    print(f"\n>>> {name} ({len(items)} goruntu x {len(RUNS)} model)...")
    PRED[name] = external_all_runs(name, items)

In [ ]:
# ── Onceki kosumun tohum-42 sonuclarini birlestir ────────────────────────
SEEDS_ALL = list(TRAIN_SEEDS)
if PREV is not None:
    added = 0
    for ds_key, ds_name in [("internal", "internal"), ("RSNA", "RSNA"), ("NIH", "NIH")]:
        for arm in ARMS:
            kt, kp = f"{ds_key}__{arm}__y_true", f"{ds_key}__{arm}__y_prob"
            if kt in PREV and kp in PREV:
                if ds_name in PRED and len(PRED[ds_name]) > 0:
                    n_new = len(next(iter(PRED[ds_name].values()))["y_true"])
                    if len(PREV[kt]) != n_new:
                        print(f"  ATLANDI {ds_name}/{arm}: n={len(PREV[kt])} != {n_new}")
                        continue
                PRED[ds_name][(SPLIT_SEED, arm)] = {"y_true": PREV[kt], "y_prob": PREV[kp]}
                added += 1
    if added:
        SEEDS_ALL = [SPLIT_SEED] + SEEDS_ALL
        print(f"Onceki kosumdan {added} dizi birlestirildi -> tohumlar: {SEEDS_ALL}")
else:
    print(f"Tohumlar: {SEEDS_ALL} (onceki kosum birlestirilmedi)")

DATASETS = [d for d in ["internal", "RSNA", "NIH"] if d in PRED and len(PRED[d])]
DS_LABEL = {"internal": "Ic test", "RSNA": "RSNA (yetiskin)", "NIH": "NIH ChestX-ray14"}
print("Degerlendirilecek kumeler:", [DS_LABEL[d] for d in DATASETS])

In [ ]:
def _midrank(x):
    J = np.argsort(x); Z = x[J]; N = len(x); T = np.zeros(N, dtype=float)
    i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5 * (i + j - 1)
        i = j
    T2 = np.empty(N, dtype=float); T2[J] = T + 1
    return T2


def _fast_delong(preds_sorted, m):
    n = preds_sorted.shape[1] - m
    pos, neg = preds_sorted[:, :m], preds_sorted[:, m:]
    k = preds_sorted.shape[0]
    tx = np.empty([k, m]); ty = np.empty([k, n]); tz = np.empty([k, m + n])
    for r in range(k):
        tx[r, :] = _midrank(pos[r, :])
        ty[r, :] = _midrank(neg[r, :])
        tz[r, :] = _midrank(preds_sorted[r, :])
    aucs = tz[:, :m].sum(axis=1) / m / n - (m + 1.0) / 2.0 / n
    v01 = (tz[:, :m] - tx) / n
    v10 = 1.0 - (tz[:, m:] - ty) / m
    sx = np.cov(v01); sy = np.cov(v10)
    if k == 1:
        sx = np.array([[float(sx)]]); sy = np.array([[float(sy)]])
    return aucs, sx / m + sy / n


def delong_test(y_true, p1, p2):
    '''Iliskili iki ROC egrisi icin AUC farki testi -> (auc1, auc2, z, p).'''
    y = np.asarray(y_true).astype(int)
    order = np.argsort(-y, kind="mergesort")
    m = int(y.sum())
    preds = np.vstack((np.asarray(p1), np.asarray(p2)))[:, order]
    aucs, cov = _fast_delong(preds, m)
    l = np.array([[1.0, -1.0]])
    var = float(l.dot(cov).dot(l.T))
    if var <= 0:
        return aucs[0], aucs[1], 0.0, 1.0
    z = float((aucs[0] - aucs[1]) / np.sqrt(var))
    return aucs[0], aucs[1], z, float(2 * (1 - sp_stats.norm.cdf(abs(z))))


def paired_bootstrap(y_true, prob_dict, n_boot=2000, seed=42):
    '''Sinif-katmanli yeniden ornekleme; ayni indeksler tum kollara uygulanir.'''
    rng = np.random.default_rng(seed)
    y = np.asarray(y_true)
    ip = np.where(y == 1)[0]; ineg = np.where(y == 0)[0]
    keys = list(prob_dict.keys())
    out = {a: np.empty(n_boot) for a in keys}
    for b in range(n_boot):
        ii = np.concatenate([rng.choice(ip, len(ip), replace=True),
                             rng.choice(ineg, len(ineg), replace=True)])
        yb = y[ii]
        for a in keys:
            fpr, tpr, _ = roc_curve(yb, np.asarray(prob_dict[a])[ii])
            out[a][b] = auc(fpr, tpr)
    return out


def mcnemar(y_true, p1, p2, thr=0.5):
    c1 = ((np.asarray(p1) >= thr).astype(int) == y_true)
    c2 = ((np.asarray(p2) >= thr).astype(int) == y_true)
    b = int(np.sum(c1 & ~c2)); c = int(np.sum(~c1 & c2))
    if b + c == 0:
        return b, c, 1.0
    return b, c, float(sp_stats.binomtest(b, b + c, 0.5).pvalue)


def expected_calibration_error(y_true, y_prob, n_bins=10):
    edges = np.linspace(0, 1, n_bins + 1); ece, N = 0.0, len(y_true)
    for i in range(n_bins):
        m = (y_prob > edges[i]) & (y_prob <= edges[i + 1])
        if m.sum() == 0:
            continue
        ece += (m.sum() / N) * abs(y_true[m].mean() - y_prob[m].mean())
    return ece


def metrics_block(y_true, y_prob, thr=0.5):
    y_pred = (y_prob >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    sens = tp / (tp + fn + 1e-9); spec = tn / (tn + fp + 1e-9)
    prec = tp / (tp + fp + 1e-9); f1 = 2 * prec * sens / (prec + sens + 1e-9)
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    return {"N": len(y_true), "AUC": auc(fpr, tpr),
            "AP": average_precision_score(y_true, y_prob),
            "Acc": float((y_pred == y_true).mean()), "F1": f1,
            "Duyarlilik": sens, "Ozgulluk": spec,
            "Brier": brier_score_loss(y_true, y_prob),
            "ECE": expected_calibration_error(y_true, y_prob)}

print("Istatistik araclari hazir (DeLong yerel olarak dogrulandi: "
      "SE = bootstrap SD, oran 1.000).")

## Gürültü tabanı ile etkinin karşılaştırılması

İki büyüklük aynı ölçekte hesaplanır:

- **Kol-içi değişkenlik** $s_{\text{tohum}}$: bir kolun farklı tohumlardaki AUC'lerinin
  standart sapması. Bu, "hiçbir şey değiştirmesek bile ne kadar oynardı" sorusunun yanıtı.
- **Kollar arası fark** $\Delta$: her tohumda ayrı ayrı hesaplanan C − A farkının ortalaması.
  Tohum-eşleştirilmiş olduğu için ortak gürültü sadeleşir.

Karar kuralı basittir: $|\Delta|$, $s_{\text{tohum}}$ ile aynı mertebedeyse etki
tohum gürültüsünden ayırt edilemez.

In [ ]:
# ── Tohum bazinda AUC tablosu ────────────────────────────────────────────
rows = []
for ds in DATASETS:
    for (seed, arm), d in PRED[ds].items():
        fpr, tpr, _ = roc_curve(d["y_true"], d["y_prob"])
        rows.append({"kume": DS_LABEL[ds], "kol": arm, "tohum": seed, "AUC": auc(fpr, tpr)})
df_seed = pd.DataFrame(rows).sort_values(["kume", "kol", "tohum"])
pd.set_option("display.float_format", lambda v: f"{v:.4f}")
print("=" * 66)
print("  TOHUM BAZINDA AUC")
print("=" * 66)
print(df_seed.pivot_table(index=["kume", "kol"], columns="tohum", values="AUC").to_string())

agg = (df_seed.groupby(["kume", "kol"])["AUC"]
       .agg(ort="mean", ss="std", n="count").reset_index())
print("\n" + "=" * 66)
print("  KOL ICI DEGISKENLIK (gurultu tabani)")
print("=" * 66)
print(agg.to_string(index=False))

In [ ]:
# ── Tohum-eslestirilmis kollar arasi fark ────────────────────────────────
PAIRS = [("lung", "raw"), ("lung", "roi"), ("roi", "raw")]
rows = []
for ds in DATASETS:
    seeds_here = sorted({s for (s, a) in PRED[ds]})
    for a, b in PAIRS:
        diffs = []
        for s in seeds_here:
            if (s, a) not in PRED[ds] or (s, b) not in PRED[ds]:
                continue
            da, db = PRED[ds][(s, a)], PRED[ds][(s, b)]
            fa, ta, _ = roc_curve(da["y_true"], da["y_prob"])
            fb, tb, _ = roc_curve(db["y_true"], db["y_prob"])
            diffs.append(auc(fa, ta) - auc(fb, tb))
        if not diffs:
            continue
        diffs = np.array(diffs)
        noise = float(agg[(agg.kume == DS_LABEL[ds]) & (agg.kol.isin([a, b]))]["ss"].mean())
        tstat, pval = (sp_stats.ttest_1samp(diffs, 0.0) if len(diffs) > 1 else (np.nan, np.nan))
        rows.append({"kume": DS_LABEL[ds], "karsilastirma": f"{a} - {b}",
                     "tohum sayisi": len(diffs), "dAUC ort": diffs.mean(),
                     "dAUC ss": diffs.std(ddof=1) if len(diffs) > 1 else np.nan,
                     "gurultu tabani": noise,
                     "etki/gurultu": abs(diffs.mean()) / noise if noise and noise > 0 else np.nan,
                     "t p": pval})
df_eff = pd.DataFrame(rows)
print("=" * 104)
print("  TOHUM-ESLESTIRILMIS KOLLAR ARASI FARK")
print("=" * 104)
print(df_eff.to_string(index=False))
print("=" * 104)
print("  'etki/gurultu' 1'in altindaysa fark, egitim rastgeleliginden ayirt edilemez.")
print("  t testi n=tohum sayisi ile yapilir; az tohumda gucu dusuktur, yalnizca yon gostergesidir.")

In [ ]:
# ── Tohum toplulugu (ensemble): egitim gurultusu giderilmis karsilastirma ─
ENS = {}
for ds in DATASETS:
    ENS[ds] = {}
    for arm in ARMS:
        ps = [PRED[ds][(s, arm)]["y_prob"] for s in sorted({x for x, _ in PRED[ds]})
              if (s, arm) in PRED[ds]]
        if not ps:
            continue
        any_key = next(k for k in PRED[ds] if k[1] == arm)
        ENS[ds][arm] = {"y_true": PRED[ds][any_key]["y_true"],
                        "y_prob": np.mean(np.vstack(ps), axis=0)}

rows = []
for ds in DATASETS:
    for arm in ARMS:
        if arm not in ENS[ds]:
            continue
        d = ENS[ds][arm]
        rows.append({"kume": DS_LABEL[ds], "kol": arm, **metrics_block(d["y_true"], d["y_prob"])})
df_ens = pd.DataFrame(rows)
print("=" * 100)
print("  TOHUM TOPLULUGU - METRIKLER")
print("=" * 100)
print(df_ens.to_string(index=False))

rows = []
for ds in DATASETS:
    yt = ENS[ds][ARMS[0]]["y_true"]
    boot = paired_bootstrap(yt, {a: ENS[ds][a]["y_prob"] for a in ARMS if a in ENS[ds]},
                            n_boot=N_BOOT, seed=SPLIT_SEED)
    for a, b in PAIRS:
        if a not in ENS[ds] or b not in ENS[ds]:
            continue
        A1, A2, z, p = delong_test(yt, ENS[ds][a]["y_prob"], ENS[ds][b]["y_prob"])
        diff = boot[a] - boot[b]
        rows.append({"kume": DS_LABEL[ds], "karsilastirma": f"{a} - {b}", "dAUC": A1 - A2,
                     "GA alt": float(np.percentile(diff, 2.5)),
                     "GA ust": float(np.percentile(diff, 97.5)),
                     "DeLong z": z, "DeLong p": p,
                     "anlamli": "EVET" if p < 0.05 else "hayir"})
df_ens_stat = pd.DataFrame(rows)
print("\n" + "=" * 104)
print("  TOPLULUKLAR ARASI ESLESTIRILMIS TEST  (egitim gurultusu buyuk olcude giderilmis)")
print("=" * 104)
print(df_ens_stat.to_string(index=False))
print("=" * 104)

In [ ]:
# ── Figur: tohum dagilimi ve gurultu tabani ──────────────────────────────
fig, axes = plt.subplots(1, len(DATASETS), figsize=(4.6 * len(DATASETS), 4.4), squeeze=False)
rng = np.random.default_rng(0)
for ax, ds in zip(axes[0], DATASETS):
    for k, arm in enumerate(ARMS):
        sub = df_seed[(df_seed.kume == DS_LABEL[ds]) & (df_seed.kol == arm)]
        if sub.empty:
            continue
        xj = k + rng.uniform(-0.08, 0.08, len(sub))
        ax.scatter(xj, sub["AUC"], s=42, color=ARM_COLOR[arm], zorder=4,
                   edgecolor="white", linewidth=1.2)
        m, sd = sub["AUC"].mean(), (sub["AUC"].std(ddof=1) if len(sub) > 1 else 0.0)
        ax.plot([k - 0.22, k + 0.22], [m, m], lw=2.4, color=ARM_COLOR[arm], zorder=5)
        if sd > 0:
            ax.plot([k, k], [m - sd, m + sd], lw=1.4, color="#2B3438", zorder=3)
        ax.text(k, m, f"  {m:.4f}", va="center", fontsize=8.2, zorder=6)
    ax.set_xticks(range(len(ARMS)))
    ax.set_xticklabels([ARM_LABEL[a].split(" · ")[0] for a in ARMS])
    ax.set_title(DS_LABEL[ds]); ax.set_ylabel("ROC-AUC")
    ax.yaxis.grid(True, alpha=.5); ax.set_axisbelow(True)
fig.suptitle("Her nokta bir eğitim tohumu — çizgi ortalama, dikey çubuk ± standart sapma",
             fontsize=11, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(WORK, "seed_fig_01_scatter.png"), dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
# ── Figur: etki mi, gurultu mu? ──────────────────────────────────────────
sub = df_eff.dropna(subset=["gurultu tabani"])
if len(sub):
    fig, ax = plt.subplots(figsize=(9.2, 2.8 + 0.44 * len(sub)))
    y = np.arange(len(sub))
    for i, (_, r) in enumerate(sub.iterrows()):
        a = r["karsilastirma"].split(" - ")[0]
        nb = r["gurultu tabani"]
        ax.barh(i, 2 * nb, left=-nb, height=0.52, color="#E3E9EA",
                edgecolor="#C3CCCE", zorder=2)
        ax.plot([r["dAUC ort"]], [i], "o", ms=8, color=ARM_COLOR[a],
                mec="white", mew=1.5, zorder=4)
        if not np.isnan(r["dAUC ss"]):
            ax.plot([r["dAUC ort"] - r["dAUC ss"], r["dAUC ort"] + r["dAUC ss"]],
                    [i, i], lw=1.8, color=ARM_COLOR[a], zorder=3)
        ax.text(0.038, i, f"{r['dAUC ort']:+.4f}   etki/gurultu = {r['etki/gurultu']:.2f}",
                va="center", fontsize=8.4)
    ax.axvline(0, color="#5A686F", lw=1.1, zorder=3)
    ax.set_yticks(y)
    ax.set_yticklabels([f"{r['kume']}\n{r['karsilastirma']}" for _, r in sub.iterrows()], fontsize=8)
    ax.set_xlabel("AUC farki")
    ax.set_xlim(-0.045, 0.075)
    ax.xaxis.grid(True, alpha=.45); ax.set_axisbelow(True); ax.invert_yaxis()
    ax.text(0, len(sub) - 0.35, "gri bant = ± tohum gurultusu", fontsize=8,
            color="#5A686F", ha="center")
    fig.suptitle("Kollar arasi fark, egitim gurultusunun disina cikiyor mu?",
                 fontsize=11, fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(WORK, "seed_fig_02_effect_vs_noise.png"), dpi=160, bbox_inches="tight")
    plt.show()

In [ ]:
# ── Kayit ────────────────────────────────────────────────────────────────
df_seed.to_csv(os.path.join(WORK, "seed_auc_by_seed.csv"), index=False)
agg.to_csv(os.path.join(WORK, "seed_noise_floor.csv"), index=False)
df_eff.to_csv(os.path.join(WORK, "seed_effect_vs_noise.csv"), index=False)
df_ens.to_csv(os.path.join(WORK, "seed_ensemble_metrics.csv"), index=False)
df_ens_stat.to_csv(os.path.join(WORK, "seed_ensemble_tests.csv"), index=False)

np.savez_compressed(
    os.path.join(WORK, "seed_predictions.npz"),
    **{f"{ds}__{s}__{a}__{k}": PRED[ds][(s, a)][k]
       for ds in DATASETS for (s, a) in PRED[ds] for k in ("y_true", "y_prob")})

with open(os.path.join(WORK, "seed_summary.json"), "w", encoding="utf-8") as f:
    json.dump({"split_seed": SPLIT_SEED, "train_seeds": SEEDS_ALL, "arms": ARMS,
               "epochs": EPOCHS_RUN, "config": CONFIG,
               "best_val_f1": {f"{s}_{a}": float(v) for (s, a), v in BESTF1.items()},
               "merged_previous_run": PREV is not None}, f, indent=2, ensure_ascii=False)

if SAVE_CHECKPOINTS:
    for (s, a), m in RUNS.items():
        torch.save({"model_state_dict": m.state_dict(), "arch": "vit_b_16", "arm": a,
                    "train_seed": s, "config": CONFIG, "class_to_idx": CLASS_TO_IDX},
                   os.path.join(WORK, f"seed{s}_vit_{a}.pth"))

print("\n" + "=" * 72)
print("  OZET")
print("=" * 72)
print(f"  Tohum sayisi: {len(SEEDS_ALL)}  {SEEDS_ALL}")
for _, r in df_eff.iterrows():
    verdict = ("gurultuden ayirt EDILEMEZ" if (np.isnan(r["etki/gurultu"]) or r["etki/gurultu"] < 1.0)
               else "gurultunun disinda")
    print(f"  {r['kume']:<20} {r['karsilastirma']:<12} dAUC={r['dAUC ort']:+.4f}  -> {verdict}")
print("=" * 72)
for f in sorted(os.listdir(WORK)):
    if f.startswith("seed"):
        print(f"  {f:<34} {os.path.getsize(os.path.join(WORK, f))/1e6:>7.2f} MB")

## Sonucun okunması

**`etki/gurultu` < 1** → kollar arası fark, aynı kolun tohumdan tohuma oynamasından
küçüktür. Makalede *"ayrıştırma başarımı bakımından fark yoktur"* denebilir ve bu artık
ölçülmüş bir ifadedir, varsayım değil.

**`etki/gurultu` > 2 ve tutarlı işaret** → etki gerçektir. Topluluk testindeki DeLong
sonucu bunu doğrulamalıdır.

**Topluluk satırları** en temiz karşılaştırmadır: tohum ortalaması eğitim gürültüsünü
büyük ölçüde giderir, geriye kalan fark ön-işlemeye atfedilebilir. Makalede hem tekil
tohumların dağılımı hem de topluluk sonucu verilmelidir.

### Raporlama önerisi

Yöntem bölümünde tek cümle yeter: *"Her kol N bağımsız eğitim tohumuyla eğitilmiş,
sonuçlar ortalama ± standart sapma olarak verilmiş, kollar arası farklar tohum-eşleştirilmiş
olarak hesaplanmıştır."* Bu, hakemin "tek koşum mu?" itirazını baştan kapatır.

> **Kısıt.** Tohum sayısı azdır (2–3). Standart sapma tahmini kendisi de gürültülüdür;
> 5 tohum ideal olurdu ancak süre maliyeti doğrusal artar. Mevcut tasarım, etkinin
> gürültüyle aynı mertebede olup olmadığını söylemeye yeter — daha ince bir kestirim
> için tohum sayısı artırılmalıdır.